# Faithfulness of the attribution maps

Measures how much the model's prediction changes when the regions an attribution map marks as most relevant are
removed from the tissue mask, compared with removing random regions. A larger gap means a more faithful explanation.

The model is a black box here: this notebook holds no model code. For each perturbed mask it calls `run(args)` from your
`inference.py` and reads the prediction that writes.

**Method**
1. The tissue mask (index 0, about 4 µm/px) is split into a grid of patches starting at (0, 0). A 256 px patch at
   0.5 µm/px is 32×32 mask pixels. Only patches with at least `PATCH_TISSUE_MIN` (10%) tissue pixels count, and
   **100% means all of those patches**.
2. Each patch takes the median of the attribution map (0-255) over its tissue pixels. Patches are ranked by that
   value, highest first, and ties are broken with one `TIE_SEED` for the whole dataset.
3. At each step, x% of the patches are removed from the mask: the top x% for the attribution order, or a random x%
   for each seed (33, 79, 98). Removed patches are set to 0 in the mask, the mask is saved, and the model runs on it.
4. y = |z_perturbed − z_unperturbed|, where z uses the mean and SD (ddof=1) of the precomputed unperturbed predictions
   of the whole dataset. The unperturbed value is the precomputed prediction.
5. AUC is the area under the curve over linear percent removed, with y = 0 at 0%. It uses the trapezoid rule: neighbouring
   points are joined by straight lines, and each segment adds its width × the average of its two heights. The area is
   divided by the x range so that it reads as the average |Δz|. **gap = AUC_attribution − AUC_random**, with the random curve averaged over the seeds. It is
   negative if the map does worse than random.

**Inputs** (set the paths in the Configuration cell)

- `CASES_DIR/<slide>.tif`: the whole-slide images
- `MASK_DIR/<slide>_tissue.tif` or `MASK_DIR/<slide>.tif`: the tissue masks, with tissue pixels set to 1
- `ATTENTION_OUTPUT_DIR/`, from your model's unperturbed run:
  - `predictions.csv`: the columns `slide_id` and `overall_survival_years`, one prediction per slide
  - `attention/<slide>/<slide>_<MAP_NAME>_attention.tif`: the attribution map, values 0-255, on the tissue mask's pixel grid
- `INFERENCE_PY`: your `inference.py`, with these parts:
  - `get_args_parser()`: returns its `argparse` parser; `MODEL_ARGS` is parsed with it
  - `run(args)`: predicts for the slide in `MODEL_WSI_DIR` with the mask in `MODEL_MASK_DIR`, and writes the prediction,
    one number in years, to `MODEL_OUTPUT_JSON`

**How the model is run.** Before each run the notebook links the slide and the perturbed mask into `MODEL_WSI_DIR` and
`MODEL_MASK_DIR`, deletes the previous `MODEL_OUTPUT_JSON` and empties `MODEL_SCRATCH_DIR/<slide>/`, so no run can see
what another left behind.
- `MODEL_WSI_DIR` and `MODEL_MASK_DIR` must be folders only this notebook fills. It stops if they hold real files.
- `MODEL_OUTPUT_JSON` must not be in `ATTENTION_OUTPUT_DIR`, or the model's runs could overwrite `predictions.csv`.
- If your `inference.py` uses fixed paths, run the notebook where those paths exist and are writable, for example inside
  your container with a writable folder bound there.
- Make one GPU visible to the job.
- `run(args)` loads the model every time, so each run is independent, and takes that much longer.
- The check before the main run repeats one slide's unperturbed prediction, and stops if it isn't the one in
  `predictions.csv`. This catches wrong `MODEL_ARGS` or missing weights before the long run.

**Outputs**

In `OUTPUT_DIR/results/<RUN_NAME>/` (`RUN_NAME` is `<MODEL_NAME>_<MAP_NAME>`):
- `runs.csv`: one row per model run, appended as soon as the run finishes
- `slide_gaps.csv`: AUCs and gaps, appended as soon as a slide is complete
- `slide_checks.csv`: patch counts and mask checks per slide
- `failed_slides.csv`: slides without a complete curve, with the reason (rewritten as the run goes)
- `last_model_run.log`: what `inference.py` printed in its latest run
- `dataset_curves.csv`, `dataset_summary.csv`: written when all slides are done
- `dataset_curves.png`

In `OUTPUT_DIR/monitor/<RUN_NAME>/`, for every slide: `<slide>_inputs.png`, `<slide>_perturbations.png`, `<slide>_curve.png`.

In `PERTURBED_MASK_DIR/tile<N>/` (`<N>` is the patch size in mask pixels; every mask has the size, spacing and levels of
its original):
- `random/seed<S>/<slide>/`: depends only on the tissue mask and the seed, so other models can reuse these masks
- `random_independent/seed<S>/<slide>/`: the same with `REMOVAL_MODE = "independent"`
- `<RUN_NAME>/<slide>/`: the attribution order's masks
- each folder holds `<slide>_tissue_pct<xx>.tif` and `manifest.csv`
- if the installed ASAP can't limit its pyramid, a mask may have an extra lowest level, which the model doesn't read

**Running on SLURM.** Run the notebook headless, with no cell timeout. If the job reaches its time limit, submit it
again: finished runs are skipped. Masks that already exist are checked pixel by pixel against what the settings give, and
reused. Start with `MAX_CASES = 3`, then look at the panels in `monitor/` before the
full run. Setting `MAX_CASES = None` afterwards continues in the same `OUTPUT_DIR`.

```bash
#!/bin/bash
#SBATCH --job-name=model-faithfulness
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=96G
#SBATCH --time=3-00:00:00
source /path/to/env/bin/activate
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 \
    --output model_name_faithfulness_blackbox.done.ipynb model_name_faithfulness_blackbox.ipynb
```

**Required libraries:** `jupyter`, `numpy`, `pandas`, `matplotlib`, `opencv-python`, `tqdm`, `wholeslidedata`, the ASAP
python bindings, and whatever your `inference.py` needs.

## Configuration

In [ ]:
import os
from pathlib import Path

# --- paths: set these to your own locations (see Inputs above) ----------------
PROJECT_DIR = Path("/path/to/project")
CASES_DIR = Path("/path/to/slides")
MASK_DIR = Path("/path/to/tissue_masks")
ATTENTION_OUTPUT_DIR = PROJECT_DIR / "model_name_outputs_attention"  # predictions.csv and attention/<slide>/
OUTPUT_DIR = PROJECT_DIR / "model_name_faithfulness_tiff"
PERTURBED_MASK_DIR = PROJECT_DIR / "perturbed_masks"     # kept outside OUTPUT_DIR: other models reuse the random masks

# --- model: your inference code, called as a black box --------------------------
INFERENCE_PY = Path("/path/to/model_code/inference.py")  # defines get_args_parser() and run(args)
MODEL_ARGS = []                                          # arguments for inference.py, the ones that produced predictions.csv
MODEL_WSI_DIR = Path("/path/to/model_input/slides")      # where inference.py looks for slides; only the notebook fills it
MODEL_MASK_DIR = Path("/path/to/model_input/tissue_masks")  # where inference.py looks for masks; only the notebook fills it
MODEL_OUTPUT_JSON = Path("/path/to/model_output/overall-survival-years.json")  # where inference.py writes its prediction
MODEL_SCRATCH_DIR = Path("/path/to/model_scratch")       # where inference.py keeps files as <dir>/<slide>/; None if it keeps none
MODEL_CHECK_MAX_DELTA_Z = 0.01  # the largest |Δz| allowed when repeating an unperturbed prediction

# --- inputs ------------------------------------------------------------------
MODEL_NAME = "model_name"
MAP_NAME = "patch"          # name of the attribution map (<slide>_<MAP_NAME>_attention.tif)
WSI_SUFFIX = ".tif"
MASK_SUFFIX = "_tissue"     # mask for <slide>.tif is <slide><MASK_SUFFIX>.tif (or <slide>.tif in MASK_DIR)
MAX_CASES = None            # e.g. 3 for a pilot; None -> every slide in predictions.csv

# --- perturbation --------------------------------------------------------------
SPACING = 0.5               # µm/px at which PATCH_SIZE is measured
PATCH_SIZE = 256            # patch side in pixels at SPACING; the size in mask pixels comes from each slide's real spacing
PATCH_TISSUE_MIN = 0.10     # a patch is tissue if at least this share of it is tissue pixels
PERCENTAGES = [1, 2, 4, 8, 16, 32, 64]  # % of the tissue patches. Add 0 for a nothing-removed check (see below)
RANDOM_SEEDS = (33, 79, 98)
N_RANDOM_SEEDS = 1          # how many of RANDOM_SEEDS to run (1 to 3)
REMOVAL_MODE = "cumulative" # random patches: "cumulative" (each step contains the previous one) or
                            # "independent" (a new draw at each step). The attribution order is always cumulative
TIE_SEED = 0                # one seed for the whole dataset; not one of RANDOM_SEEDS, or ties would follow that random order
BACKEND = "asap"            # how slides, masks and maps are read

# --- monitoring and summary ----------------------------------------------------
INLINE_SLIDES = 5     # the first slides' panels are also shown in the notebook; every slide's panels are saved
THUMB_MAX_SIZE = 1024 # max side (px) of the slide thumbnail
PANEL_SIZE = 360      # max side (px) of one panel of the perturbation grid
BOOTSTRAP_SAMPLES = 2000

assert REMOVAL_MODE in ("cumulative", "independent"), f"unknown removal mode {REMOVAL_MODE!r}"
assert 1 <= N_RANDOM_SEEDS <= len(RANDOM_SEEDS), "N_RANDOM_SEEDS must be between 1 and 3"
assert TIE_SEED not in RANDOM_SEEDS, "TIE_SEED must differ from the random seeds"
assert 0 <= min(PERCENTAGES) and max(PERCENTAGES) <= 100 and list(PERCENTAGES) == sorted(set(PERCENTAGES))
assert any(pct > 0 for pct in PERCENTAGES), "PERCENTAGES needs at least one step above 0"

RUN_NAME = f"{MODEL_NAME}_{MAP_NAME}"
SEEDS = tuple(RANDOM_SEEDS[:N_RANDOM_SEEDS])
STEPS = [pct for pct in PERCENTAGES if pct > 0]
ZERO_CHECK = 0 in PERCENTAGES  # one extra run per slide on a rewritten, unchanged mask; the curves still start at the precomputed value
RESULTS = OUTPUT_DIR / "results" / RUN_NAME
MONITOR = OUTPUT_DIR / "monitor" / RUN_NAME
RUNS_CSV = RESULTS / "runs.csv"
GAPS_CSV = RESULTS / "slide_gaps.csv"
CHECKS_CSV = RESULTS / "slide_checks.csv"
FAILED_CSV = RESULTS / "failed_slides.csv"
MODEL_LOG = RESULTS / "last_model_run.log"
print(f"{len(STEPS)} steps x (attribution + {len(SEEDS)} random seeds) = {len(STEPS) * (1 + len(SEEDS)) + ZERO_CHECK} model runs per slide")

## Environment

In [ ]:
import sys

assert INFERENCE_PY.is_file(), f"{INFERENCE_PY} not found"
# inference.py may write other files next to its prediction, such as its own predictions.csv
assert MODEL_OUTPUT_JSON.parent.resolve() != ATTENTION_OUTPUT_DIR.resolve(), (
    "ATTENTION_OUTPUT_DIR must not be the folder of MODEL_OUTPUT_JSON, or the model's runs could overwrite predictions.csv"
)

# the folder of inference.py goes first on the path, so that its own imports win over anything else installed
code_dir = str(INFERENCE_PY.resolve().parent)
if code_dir in sys.path:
    sys.path.remove(code_dir)
sys.path.insert(0, code_dir)

import inference
assert Path(inference.__file__).resolve() == INFERENCE_PY.resolve(), (
    f"`inference` was imported from {inference.__file__}, not from {INFERENCE_PY}"
)
for name in ("get_args_parser", "run"):
    assert hasattr(inference, name), f"{INFERENCE_PY} has no {name}()"
model_args = inference.get_args_parser().parse_args(MODEL_ARGS)
print(f"inference.py from {inference.__file__}, with {model_args}")

In [ ]:
import contextlib
import gc
import json
import math
import shutil
import sys
import time
import traceback

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wholeslidedata as wsd
from IPython.display import Image as DisplayImage, display
from tqdm.auto import tqdm

try:
    import multiresolutionimageinterface as mir  # ASAP bindings, also used by backend="asap"
except ImportError as e:
    raise ImportError("ASAP python bindings not found; they are needed to read and write masks") from e

for folder in (RESULTS, MONITOR, PERTURBED_MASK_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# a run's settings can't change halfway, or its rows would mix; MAX_CASES and the monitoring settings may
RUN_CONFIG = {
    "model_name": MODEL_NAME, "map_name": MAP_NAME, "patch_size": PATCH_SIZE, "patch_tissue_min": PATCH_TISSUE_MIN,
    "percentages": list(PERCENTAGES), "seeds": list(SEEDS), "removal_mode": REMOVAL_MODE, "tie_seed": TIE_SEED,
    "model_args": list(MODEL_ARGS), "attention_output_dir": str(ATTENTION_OUTPUT_DIR), "mask_dir": str(MASK_DIR),
}
config_fp = RESULTS / "config.json"
if config_fp.exists():
    previous = json.loads(config_fp.read_text())
    changed = {key: (previous.get(key), value) for key, value in RUN_CONFIG.items() if previous.get(key) != value}
    assert not changed, f"settings differ from the run in {RESULTS}: {changed}. Use another OUTPUT_DIR"
else:
    config_fp.write_text(json.dumps(RUN_CONFIG, indent=2))

## Cases and z-score statistics

The z-score statistics come from all the unperturbed predictions in `predictions.csv`, whatever `MAX_CASES` is.

In [ ]:
def find_mask(wsi_fp):
    candidates = [MASK_DIR / f"{wsi_fp.stem}{MASK_SUFFIX}{wsi_fp.suffix}", MASK_DIR / wsi_fp.name]
    return next((c for c in candidates if c.is_file()), None)


def attention_fp(slide_id):
    return ATTENTION_OUTPUT_DIR / "attention" / slide_id / f"{slide_id}_{MAP_NAME}_attention.tif"


predictions = pd.read_csv(ATTENTION_OUTPUT_DIR / "predictions.csv", dtype={"slide_id": str})
assert predictions.slide_id.is_unique, "duplicate slides in predictions.csv"
assert predictions.overall_survival_years.notna().all(), "missing predictions in predictions.csv"
Z_MEAN = float(predictions.overall_survival_years.mean())
Z_SD = float(predictions.overall_survival_years.std(ddof=1))
assert Z_SD > 0, "the unperturbed predictions have no spread, so they can't be z-scored"
print(f"{len(predictions)} unperturbed predictions: mean {Z_MEAN:.4f}, SD {Z_SD:.4f} years")


def to_z(years):
    return (years - Z_MEAN) / Z_SD


cases = predictions[["slide_id", "overall_survival_years"]].rename(columns={"overall_survival_years": "baseline_years"})
cases["wsi_path"] = [CASES_DIR / f"{slide_id}{WSI_SUFFIX}" for slide_id in cases.slide_id]
cases["mask_path"] = [find_mask(fp) for fp in cases.wsi_path]
cases["map_path"] = [attention_fp(slide_id) for slide_id in cases.slide_id]
usable = (
    cases.wsi_path.map(lambda fp: fp.is_file())
    & cases.mask_path.notna()
    & cases.map_path.map(lambda fp: fp.is_file())
)
if not usable.all():
    print("Left out, slide, mask or attribution map not found:", cases.slide_id[~usable].tolist())
cases = cases[usable].reset_index(drop=True)
if MAX_CASES is not None:
    cases = cases.head(MAX_CASES)
assert len(cases), "no slides with a slide file, a tissue mask and an attribution map"
print(f"{len(cases)} slides to evaluate")
cases.head()

## Perturbation functions

These only depend on the tissue mask, the map and the seeds. Patches are numbered row by row, so a random selection
depends only on the mask and the seed, not on the model or the order slides are processed.

In [ ]:
TISSUE_VALUE = 1  # the masks' value for tissue


def find_tissue_patches(tissue, tile, min_fraction):
    """The patches (rows, cols) of the grid starting at (0, 0) with enough tissue, and their tissue pixels."""
    h, w = tissue.shape
    counts = np.add.reduceat(
        np.add.reduceat(tissue.astype(np.int64), np.arange(0, h, tile), axis=0), np.arange(0, w, tile), axis=1
    )
    rows, cols = np.nonzero((counts > 0) & (counts >= min_fraction * tile ** 2))
    return rows, cols, counts[rows, cols]


def patch_medians(values, tissue, rows, cols, tile):
    """Each patch's median of the map over its tissue pixels."""
    medians = np.empty(len(rows), dtype=np.float64)
    for i, (r, c) in enumerate(zip(rows, cols)):
        window = (slice(r * tile, (r + 1) * tile), slice(c * tile, (c + 1) * tile))
        medians[i] = np.median(values[window][tissue[window]])
    return medians


def patches_to_remove(pct, n_patches):
    """x% of the tissue patches, rounded up; rounding first keeps e.g. 7.000000001 at 7."""
    return math.ceil(round(pct * n_patches / 100, 6))


def attribution_order(medians, tie_seed):
    """Patch indices, highest median first; a seeded shuffle first, so a stable sort breaks ties at random."""
    shuffled = np.random.default_rng(tie_seed).permutation(len(medians))
    return shuffled[np.argsort(-medians[shuffled], kind="stable")]


def random_selection(n_patches, seed, pct, mode):
    """The random patches removed at one step."""
    # independent: one draw per step. cumulative: one shuffle per slide and seed, so each step contains the previous one
    rng_seed = seed if mode == "cumulative" else [seed, int(round(pct * 1000))]
    return np.random.default_rng(rng_seed).permutation(n_patches)[:patches_to_remove(pct, n_patches)]


def remove_patches(mask, rows, cols, tile):
    """A copy of the mask with the given patches set to 0."""
    out = mask.copy()
    for r, c in zip(rows, cols):
        out[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile] = 0
    return out


def mask_folder(tile, order, slide_id):
    if order == "attribution":
        group = RUN_NAME
    elif order == "zero_check":
        group = "zero_check"
    else:  # seed<S>
        group = f"{'random' if REMOVAL_MODE == 'cumulative' else 'random_independent'}/{order}"
    return PERTURBED_MASK_DIR / f"tile{tile}" / group / slide_id


def mask_name(slide_id, pct):
    return f"{slide_id}{MASK_SUFFIX}_pct{pct:02g}.tif"

## Metric functions

In [ ]:
def curve_auc(percentages, values):
    """Trapezoid area over linear percent removed, from y = 0 at 0%, divided by the x range: the curve's average."""
    x = np.r_[0.0, np.asarray(percentages, dtype=np.float64) / 100]
    y = np.r_[0.0, np.asarray(values, dtype=np.float64)]
    return float(np.sum((x[1:] - x[:-1]) * (y[1:] + y[:-1]) / 2) / (x[-1] - x[0]))


def curve_auc_equal_steps(values):
    """The same with every step the same width, so that the last steps don't dominate."""
    y = np.r_[0.0, np.asarray(values, dtype=np.float64)]
    return float(np.sum((y[1:] + y[:-1]) / 2) / (len(y) - 1))


def slide_curves(runs, percentages, seeds):
    """The slide's |Δz| at each step for the attribution order, each seed, and the mean of the seeds; None if a step is missing."""
    by_step = runs[runs.status == "ok"].set_index(["order", "pct"]).abs_delta_z
    curves = {}
    for order in ["attribution", *[f"seed{seed}" for seed in seeds]]:
        try:
            curves[order] = np.array([by_step.loc[(order, pct)] for pct in percentages], dtype=np.float64)
        except KeyError:
            return None
    curves["random"] = np.mean([curves[f"seed{seed}"] for seed in seeds], axis=0)
    return curves


def slide_scores(curves, percentages, seeds):
    auc_attribution, auc_random = curve_auc(percentages, curves["attribution"]), curve_auc(percentages, curves["random"])
    equal_attribution, equal_random = curve_auc_equal_steps(curves["attribution"]), curve_auc_equal_steps(curves["random"])
    return {
        "auc_attribution": auc_attribution,
        "auc_random": auc_random,
        **{f"auc_seed{seed}": curve_auc(percentages, curves[f"seed{seed}"]) for seed in seeds},
        "gap": auc_attribution - auc_random,
        "auc_equal_attribution": equal_attribution,
        "auc_equal_random": equal_random,
        "gap_equal": equal_attribution - equal_random,
    }


def bootstrap_ci(values, samples, seed=0):
    """95% percentile interval of the mean, resampling slides."""
    values = np.asarray(values, dtype=np.float64)
    picks = np.random.default_rng(seed).integers(0, len(values), size=(samples, len(values)))
    return tuple(np.percentile(values[picks].mean(axis=1), [2.5, 97.5]))

## Mask reading and writing

Masks and maps are read with ASAP, as the model reads them. A mask is written with ASAP from its full resolution, then read
back: every level must have the original's shape and the expected pixels, so a mask that isn't what the model should see stops
that slide. Original masks don't all have the same number of levels, so the writer is told how many to make. Without this, a difference in the level the model reads would look like a change caused by the removal.

In [ ]:
def read_level(fp, level=0):
    image = wsd.WholeSlideImage(fp, backend=BACKEND)
    return image.get_slide(spacing=image.spacings[level])[..., 0]


def level_shapes(fp):
    return [(int(w), int(h)) for w, h in wsd.WholeSlideImage(fp, backend=BACKEND).shapes]


def _mir(group, name):
    # ASAP 2.x names constants e.g. Compression_LZW, ASAP 1.9 just LZW
    return getattr(mir, f"{group}_{name}") if hasattr(mir, f"{group}_{name}") else getattr(mir, name)


@contextlib.contextmanager
def silence_native_stdout():
    # ASAP prints its timings from C++, which contextlib.redirect_stdout doesn't catch
    sys.stdout.flush()
    saved, devnull = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 1)
    try:
        yield
    finally:
        os.dup2(saved, 1)
        os.close(saved)
        os.close(devnull)


MASK_LEVELS_REQUIRED = 3  # the model reads level 2 (about 16 µm/px), so a saved mask needs at least levels 0-2


def write_mask(path, mask, spacing, n_levels, tile_size=512):
    """Writes a uint8 mask as a pyramidal TIFF in the masks' format, under a temporary name so a stopped job leaves no partial mask."""
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(f"{path.stem}.partial.tif")
    h, w = mask.shape
    writer = mir.MultiResolutionImageWriter()
    writer.openFile(str(partial))
    writer.setTileSize(tile_size)
    writer.setCompression(_mir("Compression", "LZW"))
    writer.setDataType(_mir("DataType", "UChar"))
    writer.setInterpolation(_mir("Interpolation", "NearestNeighbor"))
    writer.setColorType(_mir("ColorType", "Monochrome"))
    if hasattr(writer, "setMaxNumberOfPyramidLevels"):  # not in every ASAP version
        writer.setMaxNumberOfPyramidLevels(n_levels)
    writer.writeImageInformation(w, h)
    spacing_vec = mir.vector_double()
    spacing_vec.push_back(spacing)
    spacing_vec.push_back(spacing)
    writer.setSpacing(spacing_vec)
    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):
            part = np.zeros((tile_size, tile_size), np.uint8)
            piece = mask[y:y + tile_size, x:x + tile_size]
            part[:piece.shape[0], :piece.shape[1]] = piece
            writer.writeBaseImagePart(part.flatten())
    with silence_native_stdout():
        writer.finishImage()
    del writer  # closes the file
    os.replace(partial, path)


def mask_file_problem(path, expected, original_shapes):
    """Why the saved mask isn't the expected mask at every level (each level the one above subsampled); None if it is."""
    shapes = level_shapes(path)
    shared = min(len(shapes), len(original_shapes))
    if shapes[:shared] != original_shapes[:shared] or len(shapes) < min(MASK_LEVELS_REQUIRED, len(original_shapes)):
        return f"levels {shapes}, the original has {original_shapes}"
    for level, (w, h) in enumerate(original_shapes[:shared]):
        factor = 2 ** level
        if not np.array_equal(read_level(path, level), expected[::factor, ::factor][:h, :w]):
            return f"level {level} differs from the expected mask"
    return None


def ensure_mask(path, expected, original_shapes, spacing):
    """Reuses a saved mask if it's the expected one, otherwise writes it. Returns True if it was written."""
    written = not path.exists()
    if written:
        write_mask(path, expected, spacing, len(original_shapes))
    problem = mask_file_problem(path, expected, original_shapes)
    if problem:
        raise RuntimeError(
            f"{path}: {problem}. "
            + ("ASAP didn't write the expected mask" if written else "It was made with other settings; use another PERTURBED_MASK_DIR")
        )
    return written


def original_pyramid_problem(mask_fp, mask, shapes):
    """If the original's lower levels aren't its full resolution subsampled, a rewritten mask differs beyond the removed patches."""
    for level, (w, h) in enumerate(shapes):
        factor = 2 ** level
        if not np.array_equal(read_level(mask_fp, level)[:h, :w], mask[::factor, ::factor][:h, :w]):
            return f"level {level} isn't level 0 subsampled"
    return None


def mask_tile_size(slide, mask):
    """PATCH_SIZE at SPACING in mask pixels, using the slide level closest to SPACING."""
    downsamples = [slide.shapes[0][0] / shape[0] for shape in slide.shapes]
    level = int(np.argmin([abs(d - SPACING / slide.spacings[0]) for d in downsamples]))
    factor = int(round(SPACING / slide.spacings[level]))
    tile = PATCH_SIZE * factor * downsamples[level] * mask.shapes[0][0] / slide.shapes[0][0]
    if abs(tile - round(tile)) > 0.01:
        raise ValueError(f"a {PATCH_SIZE} px patch is {tile:.3f} mask pixels, not a whole number")
    return int(round(tile))

## Planning one slide

Reads the mask and the map, ranks the patches, and makes sure every mask of the slide exists before the model runs.

In [ ]:
def plan_slide(case):
    """Everything about a slide's perturbations; each step is (order, pct) -> its mask and what it removes."""
    slide = wsd.WholeSlideImage(case.wsi_path, backend=BACKEND)
    mask_image = wsd.WholeSlideImage(case.mask_path, backend=BACKEND)
    tile = mask_tile_size(slide, mask_image)
    mask_spacing = float(mask_image.spacings[0])
    shapes = level_shapes(case.mask_path)

    mask = read_level(case.mask_path, 0)
    values = read_level(case.map_path, 0)
    if values.shape != mask.shape:
        raise ValueError(f"the map is {values.shape}, the mask {mask.shape}: the map must be on the mask's grid")
    tissue = mask == TISSUE_VALUE
    rows, cols, counts = find_tissue_patches(tissue, tile, PATCH_TISSUE_MIN)
    if len(rows) == 0:
        raise RuntimeError("no tissue patches")
    medians = patch_medians(values, tissue, rows, cols, tile)
    pyramid_problem = original_pyramid_problem(case.mask_path, mask, shapes)
    tissue_total = int(tissue.sum())

    selections = {("attribution", pct): attribution_order(medians, TIE_SEED)[:patches_to_remove(pct, len(rows))] for pct in STEPS}
    for seed in SEEDS:
        for pct in STEPS:
            selections[(f"seed{seed}", pct)] = random_selection(len(rows), seed, pct, REMOVAL_MODE)
    if ZERO_CHECK:
        selections[("zero_check", 0)] = np.array([], dtype=np.int64)

    steps, manifests = {}, {}
    for (order, pct), selection in tqdm(selections.items(), desc="Masks", leave=False):
        folder = mask_folder(tile, order, case.slide_id)
        path = folder / mask_name(case.slide_id, pct)
        ensure_mask(path, remove_patches(mask, rows[selection], cols[selection], tile), shapes, mask_spacing)
        step = {
            "pct": pct,
            "n_patches_total": len(rows),
            "n_patches_removed": len(selection),
            "tissue_pixels_total": tissue_total,
            "tissue_pixels_removed": int(counts[selection].sum()),
            "tissue_removed_pct": 100 * float(counts[selection].sum()) / tissue_total,
        }
        steps[(order, pct)] = {**step, "mask_path": path, "selection": selection}
        manifests.setdefault(folder, []).append({
            "slide_id": case.slide_id, "order": order, **step, "patch_size": PATCH_SIZE, "tile_mask_pixels": tile,
            "patch_tissue_min": PATCH_TISSUE_MIN, "removal_mode": REMOVAL_MODE if order.startswith("seed") else "cumulative",
            "seed": int(order[4:]) if order.startswith("seed") else (TIE_SEED if order == "attribution" else ""),
            "mask_file": path.name,
        })
    for folder, rows_ in manifests.items():
        pd.DataFrame(rows_).to_csv(folder / "manifest.csv", index=False)

    level = next((i for i, (sw, sh) in enumerate(slide.shapes) if max(sw, sh) <= THUMB_MAX_SIZE), len(slide.shapes) - 1)
    thumb = np.asarray(slide.get_slide(spacing=slide.spacings[level]))[..., :3]
    grid_shape = (math.ceil(mask.shape[0] / tile), math.ceil(mask.shape[1] / tile))
    small = lambda image: cv2.resize(  # noqa: E731
        image.astype(np.float32), thumb.shape[1::-1], interpolation=cv2.INTER_AREA
    )
    return {
        "slide_id": case.slide_id,
        "baseline_years": float(case.baseline_years),
        "tile": tile,
        "thumb": thumb,
        "tissue_small": small(tissue),
        "map_small": small(values),
        "grid_shape": grid_shape,
        "rows": rows,
        "cols": cols,
        "medians": medians,
        "steps": steps,
        "checks": {
            "slide_id": case.slide_id,
            "tile_mask_pixels": tile,
            "n_patches": len(rows),
            "tissue_pixels": tissue_total,
            "tissue_in_patches_pct": 100 * float(counts.sum()) / tissue_total,
            "patches_with_zero_attribution_pct": 100 * float((medians == 0).mean()),
            "distinct_median_values": int(len(np.unique(medians))),
            "pyramid_problem": pyramid_problem or "",
        },
    }

## Model run

The model only runs through `run_model`, which calls `inference.run` on one slide and one mask and returns the prediction.
Every run starts clean (see "How the model is run" at the top). A run that raises an error, or leaves no finite prediction
in `MODEL_OUTPUT_JSON`, counts as failed.

In [ ]:
def empty_model_input(folder):
    """Removes the previous run's links; stops if the folder holds anything else, so real files are never deleted."""
    folder.mkdir(parents=True, exist_ok=True)
    for fp in folder.iterdir():
        if not fp.is_symlink():
            raise RuntimeError(f"{fp} isn't a link from this notebook. Point MODEL_WSI_DIR and MODEL_MASK_DIR at folders only the notebook fills")
        fp.unlink()


def link_model_input(wsi_fp, mask_fp):
    """The model sees one slide and one mask, under the original names whichever mask this is."""
    for folder in (MODEL_WSI_DIR, MODEL_MASK_DIR):
        empty_model_input(folder)
    (MODEL_WSI_DIR / wsi_fp.name).symlink_to(wsi_fp.resolve())
    (MODEL_MASK_DIR / f"{wsi_fp.stem}{MASK_SUFFIX}{mask_fp.suffix}").symlink_to(mask_fp.resolve())


def read_prediction():
    years = float(json.loads(MODEL_OUTPUT_JSON.read_text()))
    if not math.isfinite(years):
        raise RuntimeError(f"the prediction in {MODEL_OUTPUT_JSON} is {years}")
    return years


def run_model(wsi_fp, mask_fp):
    """One independent run of inference.py through one mask; returns the predicted years."""
    scratch = MODEL_SCRATCH_DIR / wsi_fp.stem if MODEL_SCRATCH_DIR else None
    link_model_input(wsi_fp, mask_fp)
    MODEL_OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    MODEL_OUTPUT_JSON.unlink(missing_ok=True)  # a run that writes nothing must not read the previous run's prediction
    if scratch:
        shutil.rmtree(scratch, ignore_errors=True)
    try:
        with open(MODEL_LOG, "w") as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            inference.run(model_args)
        return read_prediction()
    finally:
        if scratch:
            shutil.rmtree(scratch, ignore_errors=True)
        gc.collect()

## Model check

Runs the model once on the first slide with its original mask. The result must match that slide's prediction in
`predictions.csv`. If it doesn't, look at `last_model_run.log`, and at `MODEL_ARGS` and the model's weights.

In [ ]:
first_case = cases.iloc[0]
repeated_years = run_model(Path(first_case.wsi_path), Path(first_case.mask_path))
repeated_delta_z = abs(to_z(repeated_years) - to_z(first_case.baseline_years))
print(f"{first_case.slide_id}: {repeated_years:.4f} years now, {first_case.baseline_years:.4f} in predictions.csv, |Δz| {repeated_delta_z:.4f}")
assert repeated_delta_z <= MODEL_CHECK_MAX_DELTA_Z, (
    f"inference.py doesn't reproduce predictions.csv on {first_case.slide_id}: |Δz| is {repeated_delta_z:.4f}, "
    f"the limit is {MODEL_CHECK_MAX_DELTA_Z}. See {MODEL_LOG}"
)

## Monitoring panels

Each slide gets three PNGs in `monitor/`. The perturbation grid is saved before the model runs, so the masks can be
checked straight away, and saved again with each run's |Δz| when the slide is done. In the grid, colour marks
removed patches, the tissue keeps its colours, and the background is faded.

In [ ]:
SURFACE = "#fcfcfb"
PRIMARY_INK = "#0b0b0b"
SECONDARY_INK = "#52514e"
GRID = "#e4e3df"
ATTRIBUTION_COLOUR = "#2a78d6"
SEED_COLOURS = ("#eb6834", "#1baf7a", "#c3ed00")  # in the order of RANDOM_SEEDS, so a seed keeps its colour


def order_colour(order):
    return ATTRIBUTION_COLOUR if order == "attribution" else SEED_COLOURS[RANDOM_SEEDS.index(int(order[4:]))]


def hex_rgb(colour):
    return np.array([int(colour[i:i + 2], 16) for i in (1, 3, 5)], dtype=np.float32)


def overlay_panel(plan, selection, colour, size=PANEL_SIZE):
    """The slide thumbnail with the background faded and each removed patch's share of a pixel in colour."""
    height, width = plan["thumb"].shape[:2]
    scale = min(1.0, size / max(height, width))
    shape = (max(1, round(width * scale)), max(1, round(height * scale)))
    thumb = cv2.resize(plan["thumb"], shape, interpolation=cv2.INTER_AREA).astype(np.float32)
    tissue = cv2.resize(plan["tissue_small"], shape, interpolation=cv2.INTER_AREA)[..., None]
    grid = np.zeros(plan["grid_shape"], np.float32)
    grid[plan["rows"][selection], plan["cols"][selection]] = 1
    removed = cv2.resize(grid, shape, interpolation=cv2.INTER_AREA)[..., None]
    faded = thumb * (0.25 + 0.75 * tissue) + 255 * 0.75 * (1 - tissue)
    return np.clip(faded * (1 - 0.8 * removed) + hex_rgb(colour) * 0.8 * removed, 0, 255).astype(np.uint8)


def order_label(order):
    return "High attribution" if order == "attribution" else f"Random, seed {order[4:]}"


def save_figure(figure, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=170, facecolor=SURFACE)
    plt.close(figure)
    return path


def plot_inputs(plan, path):
    """Slide, tissue mask, attribution map and median per patch, to check they line up."""
    medians = np.full(plan["grid_shape"], np.nan)
    medians[plan["rows"], plan["cols"]] = plan["medians"]
    map_shown = np.where(plan["tissue_small"] > 0, plan["map_small"], np.nan)
    figure, axes = plt.subplots(1, 4, figsize=(16, 4.6), facecolor=SURFACE, layout="constrained")
    axes[0].imshow(plan["thumb"])
    axes[1].imshow(plan["tissue_small"], cmap="Greys", vmin=0, vmax=1)
    axes[2].imshow(map_shown, cmap="Blues", vmin=0, vmax=255)
    shown = axes[3].imshow(medians, cmap="Blues", vmin=0, vmax=255, interpolation="nearest")
    titles = ("Slide", "Tissue mask", "Attribution map, on tissue", f"Median per patch ({plan['tile']} mask px)")
    for ax, title in zip(axes, titles):
        ax.set_title(title, color=SECONDARY_INK, fontsize=11)
        ax.set_axis_off()
    figure.colorbar(shown, ax=axes[3], shrink=0.7, label="Attribution (0-255)")
    checks = plan["checks"]
    figure.suptitle(
        f"{plan['slide_id']}: {plan['baseline_years']:.3f} years unperturbed, {checks['n_patches']:,} tissue patches "
        f"holding {checks['tissue_in_patches_pct']:.1f}% of the tissue, {checks['patches_with_zero_attribution_pct']:.1f}% of them with attribution 0",
        color=PRIMARY_INK, fontsize=12,
    )
    return save_figure(figure, path)


def plot_perturbations(plan, path, runs=None):
    """One row per order, one column per step: the patches removed and, once run, the change in prediction."""
    orders = ["attribution", *[f"seed{seed}" for seed in SEEDS]]
    height, width = plan["thumb"].shape[:2]
    panel_w = 2.2
    panel_h = max(0.9, panel_w * height / width) + (0.45 if runs is not None else 0.32)
    figure, axes = plt.subplots(
        len(orders), len(STEPS), figsize=(panel_w * len(STEPS) + 0.5, panel_h * len(orders) + 0.5),
        facecolor=SURFACE, layout="constrained", squeeze=False,
    )
    by_step = {} if runs is None else {(row.order, row.pct): row for row in runs.itertuples()}
    for i, order in enumerate(orders):
        colour = order_colour(order)
        for j, pct in enumerate(STEPS):
            ax, step = axes[i, j], plan["steps"][(order, pct)]
            ax.imshow(overlay_panel(plan, step["selection"], colour), interpolation="nearest")
            ax.set_axis_off()
            title = f"{pct:g}% patches, {step['tissue_removed_pct']:.1f}% tissue"
            run = by_step.get((order, pct))
            if run is not None:
                title += f"\n|Δz| {run.abs_delta_z:.3f}" if run.status == "ok" else f"\n{run.status}"
            ax.set_title(title, color=SECONDARY_INK, fontsize=7)
            if j == 0:
                ax.text(-0.05, 0.5, order_label(order), transform=ax.transAxes, rotation=90, ha="right", va="center",
                        color=PRIMARY_INK, fontsize=9)
    figure.suptitle(
        f"{plan['slide_id']}: patches removed from the tissue mask ({plan['checks']['n_patches']:,} tissue patches)"
        + ("" if runs is not None else ", before the model runs"),
        color=PRIMARY_INK, fontsize=12,
    )
    return save_figure(figure, path)


def style_axes(ax, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=SECONDARY_INK, length=0, labelsize=9)
    ax.set_xlabel(xlabel, color=SECONDARY_INK, fontsize=10)
    ax.set_ylabel(ylabel, color=SECONDARY_INK, fontsize=10)


def plot_curves(curves, percentages, title, path, bands=None):
    """Absolute risk change against % perturbed: the high attribution curve and each random curve."""
    xs = np.r_[0.0, np.asarray(percentages, dtype=np.float64)]
    orders = ["attribution", *[name for name in curves if name.startswith("seed")]]
    figure, ax = plt.subplots(figsize=(9, 5), facecolor=SURFACE, layout="constrained")
    for i, order in enumerate(reversed(orders)):  # the high attribution curve on top
        colour = order_colour(order)
        if bands is not None:  # thin bars, shifted a little per curve so that they don't hide each other
            low, high = bands[order]
            shift = (i - (len(orders) - 1) / 2) * 0.004 * xs[-1]
            ax.vlines(xs[1:] + shift, low, high, color=colour, linewidth=1, alpha=0.75, zorder=2, clip_on=False)
        ax.plot(xs, np.r_[0.0, curves[order]], color=colour, linewidth=2, marker="o", markersize=5.5,
                markeredgecolor=SURFACE, markeredgewidth=1, zorder=3, clip_on=False, label=order_label(order))
    upper = max(np.nanmax(curves[order]) for order in orders)
    if bands is not None:
        upper = max(upper, *(np.nanmax(bands[order][1]) for order in orders))
    upper = max(upper * 1.08, 1e-3)
    ax.set_ylim(0, upper)
    ax.set_xlim(0, xs[-1])
    style_axes(ax, "Tissue patches perturbed (%)", "Absolute risk change, |Δz|")

    # labels beside the last points, pushed apart where curves end close together
    by_end = sorted(orders, key=lambda order: curves[order][-1])
    position = -1.0
    for order in by_end:
        position = max(position + 0.06, curves[order][-1] / upper)
        ax.plot([1.025], [position], transform=ax.transAxes, marker="o", markersize=5.5, color=order_colour(order), clip_on=False)
        ax.text(1.04, position, order_label(order), transform=ax.transAxes, va="center", color=PRIMARY_INK, fontsize=9)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = [h for h, l in zip(handles, labels) if not l.startswith("_")], [l for l in labels if not l.startswith("_")]
    figure.legend(handles[::-1], labels[::-1], loc="outside lower center", ncols=len(orders), frameon=False, labelcolor=SECONDARY_INK, fontsize=9)
    figure.suptitle(title, color=PRIMARY_INK, fontsize=12)
    return save_figure(figure, path)

## Run

Results are written as soon as they exist: a row in `runs.csv` after each run, a row in `slide_gaps.csv` after each
slide. Rerunning this cell skips finished runs and retries failed ones. A slide with a failed run, for example a step
that leaves the model nothing to read, has no gap and is listed in `failed_slides.csv`.

In [ ]:
RUN_COLUMNS = [
    "slide_id", "order", "seed", "pct", "status", "years", "z", "baseline_years", "baseline_z", "abs_delta_z",
    "n_patches_total", "n_patches_removed", "tissue_pixels_total", "tissue_pixels_removed", "tissue_removed_pct",
    "runtime_s", "mask_path", "error",
]


def append_row(path, row, columns=None):
    frame = pd.DataFrame([row], columns=columns or list(row))
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def load_runs():
    if not RUNS_CSV.exists():
        return pd.DataFrame(columns=RUN_COLUMNS)
    runs = pd.read_csv(RUNS_CSV, dtype={"slide_id": str, "order": str})
    return runs.drop_duplicates(["slide_id", "order", "pct"], keep="last")  # a retried run replaces the failure


def run_step(case, order, step):
    """One model run, written to runs.csv straight away."""
    row = {
        "slide_id": case.slide_id,
        "order": order,
        "seed": int(order[4:]) if order.startswith("seed") else None,
        "baseline_years": float(case.baseline_years),
        "baseline_z": to_z(float(case.baseline_years)),
        "mask_path": str(step["mask_path"]),
        **{key: step[key] for key in ("pct", "n_patches_total", "n_patches_removed", "tissue_pixels_total", "tissue_pixels_removed", "tissue_removed_pct")},
    }
    start = time.time()
    try:
        years = run_model(Path(case.wsi_path), step["mask_path"])
        row.update(status="ok", years=years, z=to_z(years), abs_delta_z=abs(to_z(years) - row["baseline_z"]),
                   runtime_s=round(time.time() - start, 1))
    except Exception:
        row.update(status="failed", error=traceback.format_exc())
        print(f"FAILED {case.slide_id} {order} {step['pct']}%:\n{row['error']}")
    append_row(RUNS_CSV, row, RUN_COLUMNS)
    return row


def score_slide(case, runs):
    """Writes the slide's row in slide_gaps.csv if its curves are complete; returns the row and curves, or None."""
    curves = slide_curves(runs, STEPS, SEEDS)
    if curves is None:
        return None, None
    row = {"slide_id": case.slide_id, "baseline_years": float(case.baseline_years), "baseline_z": to_z(float(case.baseline_years)),
           **slide_scores(curves, STEPS, SEEDS)}
    if ZERO_CHECK:
        zero = runs[(runs.order == "zero_check") & (runs.status == "ok")].abs_delta_z
        row["zero_check_abs_delta_z"] = float(zero.iloc[0]) if len(zero) else None
    append_row(GAPS_CSV, row)
    return row, curves


def monitor_path(slide_id, panel):
    return MONITOR / f"{slide_id}_{panel}.png"


gaps = pd.read_csv(GAPS_CSV, dtype={"slide_id": str}) if GAPS_CSV.exists() else pd.DataFrame(columns=["slide_id", "gap"])
scored = set(gaps.slide_id)
failed_slides = {}
session_runtimes = []
total_runs = len(cases) * (len(STEPS) * (1 + len(SEEDS)) + ZERO_CHECK)
progress_handle = display(pd.DataFrame(), display_id=True)


def show_progress(current):
    runs = load_runs()
    done_runs = int(runs.slide_id.isin(cases.slide_id).sum())
    per_run = float(np.mean(session_runtimes)) if session_runtimes else float("nan")
    gap_rows = pd.read_csv(GAPS_CSV, dtype={"slide_id": str}) if GAPS_CSV.exists() else gaps
    gap_rows = gap_rows[gap_rows.slide_id.isin(cases.slide_id)].drop_duplicates("slide_id", keep="last")
    table = pd.DataFrame([{
        "current slide": current,
        "slides done": f"{len(gap_rows)} / {len(cases)}",
        "failed slides": len(failed_slides),
        "model runs done": f"{done_runs} / {total_runs}",
        "min per run (this session)": round(per_run / 60, 2),
        "hours left, at that rate": round((total_runs - done_runs) * per_run / 3600, 1),
        "mean gap so far": round(float(gap_rows.gap.mean()), 4) if len(gap_rows) else None,
    }])
    if progress_handle is not None:
        progress_handle.update(table)
    return table


for index, case in enumerate(tqdm(list(cases.itertuples()), desc="Slides")):
    if case.slide_id in scored:
        continue
    show_inline = index < INLINE_SLIDES
    try:
        show_progress(case.slide_id)
        plan = plan_slide(case)
        checks = plan["checks"]
        if checks["pyramid_problem"]:
            print(f"WARNING {case.slide_id}: {checks['pyramid_problem']}; its masks differ from the original beyond the removed patches")
        previous_checks = pd.read_csv(CHECKS_CSV, dtype={"slide_id": str}).slide_id.tolist() if CHECKS_CSV.exists() else []
        if case.slide_id not in previous_checks:
            append_row(CHECKS_CSV, checks)

        inputs_png = plot_inputs(plan, monitor_path(case.slide_id, "inputs"))
        grid_png = plot_perturbations(plan, monitor_path(case.slide_id, "perturbations"))
        if show_inline:
            display(DisplayImage(filename=str(inputs_png)), DisplayImage(filename=str(grid_png)))

        finished = load_runs()
        finished = finished[(finished.slide_id == case.slide_id) & (finished.status == "ok")]
        finished = set(zip(finished.order, finished.pct))
        # masks in order of tissue removed, so that a slide's early steps are done first
        for (order, pct), step in tqdm(sorted(plan["steps"].items(), key=lambda item: item[0][1]), desc=case.slide_id, leave=False):
            if (order, pct) in finished:
                continue
            row = run_step(case, order, step)
            if "runtime_s" in row:
                session_runtimes.append(row["runtime_s"])
            show_progress(case.slide_id)

        slide_runs = load_runs()
        slide_runs = slide_runs[slide_runs.slide_id == case.slide_id]
        plot_perturbations(plan, grid_png, slide_runs)
        score, curves = score_slide(case, slide_runs)
        if score is None:
            bad = slide_runs[slide_runs.status != "ok"]
            failed_slides[case.slide_id] = "steps without a prediction: " + ", ".join(
                f"{row.order} {row.pct:g}% ({row.status})" for row in bad.itertuples()
            )
        else:
            curve_png = plot_curves(
                curves, STEPS, f"{case.slide_id}: gap {score['gap']:+.3f} (AUC attribution {score['auc_attribution']:.3f}, random {score['auc_random']:.3f})",
                monitor_path(case.slide_id, "curve"),
            )
            if show_inline:
                display(DisplayImage(filename=str(grid_png)), DisplayImage(filename=str(curve_png)))
    except Exception:
        failed_slides[case.slide_id] = traceback.format_exc()
        print(f"FAILED {case.slide_id}:\n{failed_slides[case.slide_id]}")
    pd.DataFrame(
        [{"slide_id": slide, "reason": reason} for slide, reason in failed_slides.items()], columns=["slide_id", "reason"]
    ).to_csv(FAILED_CSV, index=False)

show_progress("done")

## Dataset results

Written when the run is complete, and can be rerun at any point to see the slides done so far. The plot shows the mean
curve of the high attribution order and of each random seed, with bars for the 95% bootstrap intervals over slides. The mean of the seeds
is in `dataset_curves.csv` and is what the gap uses.

In [ ]:
def dataset_results():
    gaps = pd.read_csv(GAPS_CSV, dtype={"slide_id": str}).drop_duplicates("slide_id", keep="last")
    gaps = gaps[gaps.slide_id.isin(cases.slide_id)]
    runs = load_runs()
    runs = runs[runs.slide_id.isin(gaps.slide_id) & (runs.status == "ok") & (runs.pct > 0)]

    table = runs.pivot_table(index="slide_id", columns=["order", "pct"], values="abs_delta_z")
    per_order = {order: table[order] for order in ("attribution", *[f"seed{seed}" for seed in SEEDS])}
    per_order["random"] = sum(per_order[f"seed{seed}"] for seed in SEEDS) / len(SEEDS)
    rows, curves, bands = [], {}, {}
    for order, frame in per_order.items():
        curves[order] = frame[STEPS].mean(axis=0).to_numpy()
        intervals = [bootstrap_ci(frame[pct].to_numpy(), BOOTSTRAP_SAMPLES) for pct in STEPS]
        if order != "random":
            bands[order] = (np.array([low for low, _ in intervals]), np.array([high for _, high in intervals]))
        for pct, mean, (low, high) in zip(STEPS, curves[order], intervals):
            rows.append({"order": order, "pct": pct, "n_slides": len(frame), "mean_abs_delta_z": mean, "ci_low": low, "ci_high": high})
    pd.DataFrame(rows).to_csv(RESULTS / "dataset_curves.csv", index=False)

    metrics = ["gap", "auc_attribution", "auc_random", *[f"auc_seed{seed}" for seed in SEEDS],
               "gap_equal", "auc_equal_attribution", "auc_equal_random"]
    summary = []
    for metric in metrics:
        low, high = bootstrap_ci(gaps[metric].to_numpy(), BOOTSTRAP_SAMPLES)
        summary.append({"metric": metric, "n_slides": len(gaps), "mean": gaps[metric].mean(), "ci_low": low, "ci_high": high,
                        "median": gaps[metric].median(), "sd": gaps[metric].std(ddof=1) if len(gaps) > 1 else None})
    summary = pd.DataFrame(summary)
    summary.to_csv(RESULTS / "dataset_summary.csv", index=False)

    gap = summary.set_index("metric").loc["gap"]
    figure_fp = plot_curves(
        curves, STEPS,
        f"{RUN_NAME}, {len(gaps)} slides: mean gap {gap['mean']:+.3f} (95% CI {gap['ci_low']:+.3f} to {gap['ci_high']:+.3f})",
        RESULTS / "dataset_curves.png", bands=bands,
    )
    return gaps, summary, figure_fp


if GAPS_CSV.exists():
    slide_gaps, dataset_summary, dataset_png = dataset_results()
    print(f"{len(slide_gaps)} of {len(cases)} slides have a gap; the share with a positive gap is {(slide_gaps.gap > 0).mean():.0%}")
    if failed_slides:
        print(f"{len(failed_slides)} slides left out, see {FAILED_CSV}")
    display(DisplayImage(filename=str(dataset_png)))
    display(dataset_summary.round(4))
else:
    print("No slide has a complete curve yet")